In [ ]:
%useLatestDescriptors
%use dataframe(1.0.0-Beta5n)
%use kandy
%use serialization

In [ ]:
@file:Repository("https://repo.gradle.org/gradle/libs-releases") //
@file:DependsOn("org.gradle:gradle-tooling-api:9.6.0")

import org.gradle.tooling.GradleConnector
import java.io.File

GradleConnector.newConnector().forProjectDirectory(File("./../../")).connect().use { connection ->
    connection.newBuild()
            .forTasks(
                "clean",
                "crcBenchmark",
                "deflateBenchmark",
                "inflateBenchmark"
            )
            .setStandardOutput(System.out)
            .setStandardError(System.err)
            .run()
}

In [7]:
@file:OptIn(ExperimentalSerializationApi::class)

import kotlinx.serialization.ExperimentalSerializationApi
import kotlinx.serialization.SerialName
import kotlinx.serialization.Serializable
import kotlinx.serialization.json.Json
import kotlinx.serialization.json.decodeFromStream
import org.jetbrains.kotlinx.kandy.dsl.plot
import org.jetbrains.kotlinx.kandy.letsplot.layers.bars
import java.nio.file.Path
import kotlin.io.path.PathWalkOption
import kotlin.io.path.extension
import kotlin.io.path.inputStream
import kotlin.io.path.name
import kotlin.io.path.nameWithoutExtension
import kotlin.io.path.walk

fun String.suffixSimilarity(other: String): Double {
    val a = this.reversed()
    val b = other.reversed()
    val maxLen = maxOf(a.length, b.length)
    if (maxLen == 0) return 1.0
    var match = 0
    for (i in 0 until minOf(a.length, b.length)) {
        if (a[i] == b[i]) match++ else break
    }
    return match.toDouble() / maxLen
}

inline fun <T> List<T>.groupBySimilarityChain(crossinline selector: (T) -> String): List<T> {
    if (isEmpty()) return emptyList()
    val unused = toMutableSet()
    val result = ArrayList<T>()
    var current = unused.first()
    result += current
    unused -= current
    while (unused.isNotEmpty()) {
        val next = unused.maxBy { selector(it).suffixSimilarity(selector(current)) }
        result += next
        unused -= next
        current = next
    }
    return result
}

@Serializable
data class PrimaryMetric(
    val score: Double,
    val scoreError: Double,
    val scoreUnit: String
)

@Serializable
data class Benchmark(
    @SerialName("benchmark") val name: String,
    val warmupIterations: Int,
    val measurementIterations: Int,
    val primaryMetric: PrimaryMetric
) {
    inline val cleanName: String
        get() = name.substringBeforeLast('.').substringAfterLast('.')
}

val json: Json = Json { ignoreUnknownKeys = true }

Path.of("./../build/reports/benchmarks")
        .walk()
        .filter { filePath -> filePath.extension == "json" }
        .map { filePath ->
            filePath.inputStream().use { stream ->
                filePath to json.decodeFromStream<List<Benchmark>>(stream)
            }
        }.map { (filePath, report) ->
            val sortedReport = report.groupBySimilarityChain(Benchmark::name)
            val plotName = filePath.parent.parent.name
            val platform = filePath.nameWithoutExtension
            val df = dataFrameOf(
                "name" to sortedReport.map(Benchmark::cleanName),
                "score" to sortedReport.map { benchmark -> benchmark.primaryMetric.score },
                "score_min" to sortedReport.map { benchmark -> benchmark.primaryMetric.score - benchmark.primaryMetric.scoreError },
                "score_max" to sortedReport.map { benchmark -> benchmark.primaryMetric.score + benchmark.primaryMetric.scoreError }
            )
            val plot = plot(df) {
                layout {
                    size = 1200 to 800
                    title = "$plotName ($platform)" // Parent of parent is suite name
                    xAxisLabel = "Implementation"
                    yAxisLabel = "MiB/s"
                    theme = Theme.DARCULA
                }
                bars {
                    x("name")
                    y("score") {
                        axis {
                            breaks(format = ".2f")
                        }
                    }
                    fillColor("name")
                }
                errorBars {
                    x("name")
                    yMin("score_min")
                    yMax("score_max")
                }
            }
            plot.save("./../../docs/${plotName}_$platform.png")
            plot
        }.forEach(::DISPLAY)

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="wp1sE2" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1200.0, 
 height: 800.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("wp1sE2");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"deflate (linuxX64)"
},
"mapping":{
},
"guides":{
"x":{
"title":"Implementation"
},
"y":{
"title":"MiB/s"
}
},
"data":{
"score":[88.14562496951662,219.58332719198182,88.5735324496175,161.8619194075953,90.92790560547768,825.5971003563398],
"name":["DeflaterDefaultLevelBenchmark","NativeDeflaterDefaultLevelBenchmark","DeflaterMaxLevelBenchmark","NativeDeflaterMaxLevelBenchmark","DeflaterMinLevelBenchmark","NativeDeflaterMinLevelBenchmark"],
"score_max":[88.8416576862866,221.59688470022755,89.30769890932439,168.58716917537285,92.0446795539639,830.4439208849436],
"score_min":[87.44959225274664,217.5697696837361,87.83936598991062,155.13666963981777,89.81113165699145,820.7502798277359]
},
"ggsize":{
"width":1200.0,
"height":800.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"format":".2f",
"limits":[null,null]
},{
"aesthetic":"fill",
"discrete":true
},{
"aesthetic":"x",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"name",
"y":"score",
"fill":"name"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data":{
}
},{
"mapping":{
"x":"name",
"ymin":"score_min",
"ymax":"score_max"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"errorbar",
"data":{
}
}],
"theme":{
"flavor":"darcula"
},
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"name"
},{
"type":"float",
"column":"score"
},{
"type":"float",
"column":"score_min"
},{
"type":"float",
"column":"score_max"
}]
},
"spec_id":"175"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 DeflaterDefaultLevelBenchmark 
 
 
 
 
 
 
 
 
 NativeDeflaterDefaultLevelBenchmark 
 
 
 
 
 
 
 
 
 DeflaterMaxLevelBenchmark 
 
 
 
 
 
 
 
 
 NativeDeflaterMaxLevelBenchmark 
 
 
 
 
 
 
 
 
 DeflaterMinLevelBenchmark 
 
 
 
 
 
 
 
 
 NativeDeflaterMinLevelBenchmark 
 
 
 
 
 
 
 
 
 
 
 0.00 
 
 
 
 
 
 
 100.00 
 
 
 
 
 
 
 200.00 
 
 
 
 
 
 
 300.00 
 
 
 
 
 
 
 400.00 
 
 
 
 
 
 
 500.00 
 
 
 
 
 
 
 600.00 
 
 


<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="CnD3VL" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1200.0, 
 height: 800.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("CnD3VL");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"deflate (js)"
},
"mapping":{
},
"guides":{
"x":{
"title":"Implementation"
},
"y":{
"title":"MiB/s"
}
},
"data":{
"score":[70.1168514926414,69.10609686358319,68.33032927617498],
"name":["DeflaterDefaultLevelBenchmark","DeflaterMaxLevelBenchmark","DeflaterMinLevelBenchmark"],
"score_max":[70.9273458263167,69.80296622044678,68.71554356139418],
"score_min":[69.30635715896611,68.4092275067196,67.94511499095577]
},
"ggsize":{
"width":1200.0,
"height":800.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"format":".2f",
"limits":[null,null]
},{
"aesthetic":"fill",
"discrete":true
},{
"aesthetic":"x",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"name",
"y":"score",
"fill":"name"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data":{
}
},{
"mapping":{
"x":"name",
"ymin":"score_min",
"ymax":"score_max"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"errorbar",
"data":{
}
}],
"theme":{
"flavor":"darcula"
},
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"name"
},{
"type":"float",
"column":"score"
},{
"type":"float",
"column":"score_min"
},{
"type":"float",
"column":"score_max"
}]
},
"spec_id":"179"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 DeflaterDefaultLevelBenchmark 
 
 
 
 
 
 
 
 
 DeflaterMaxLevelBenchmark 
 
 
 
 
 
 
 
 
 DeflaterMinLevelBenchmark 
 
 
 
 
 
 
 
 
 
 
 0.00 
 
 
 
 
 
 
 5.00 
 
 
 
 
 
 
 10.00 
 
 
 
 
 
 
 15.00 
 
 
 
 
 
 
 20.00 
 
 
 
 
 
 
 25.00 
 
 
 
 
 
 
 30.00 
 
 
 
 
 
 
 35.00 
 
 
 
 
 
 
 40.00 
 
 
 
 
 
 
 45.00 
 
 
 
 
 
 
 50.00 
 
 
 
 
 
 
 55.00 
 
 
 
 
 
 
 60.00 
 
 
 
 
 
 
 65.00 
 
 
 
 
 
 
 70.00 
 
 
 
 
 
 
 
 
 deflate (js) 
 
 
 
 
 MiB/s 
 
 
 
 
 Implementation 
 
 
 
 
 
 
 
 
 name 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 DeflaterDefaultLevelBenchmark 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 DeflaterMaxLevelBenchmark 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 DeflaterMinLevelBenchmark

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="lqGiYO" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1200.0, 
 height: 800.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("lqGiYO");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"deflate (wasmJs)"
},
"mapping":{
},
"guides":{
"x":{
"title":"Implementation"
},
"y":{
"title":"MiB/s"
}
},
"data":{
"score":[83.83357066282706,81.49406361216965,83.3591813765535],
"name":["DeflaterDefaultLevelBenchmark","DeflaterMaxLevelBenchmark","DeflaterMinLevelBenchmark"],
"score_max":[84.4400166154508,83.37811374451192,84.12346508900109],
"score_min":[83.22712471020331,79.61001347982739,82.5948976641059]
},
"ggsize":{
"width":1200.0,
"height":800.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"format":".2f",
"limits":[null,null]
},{
"aesthetic":"fill",
"discrete":true
},{
"aesthetic":"x",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"name",
"y":"score",
"fill":"name"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data":{
}
},{
"mapping":{
"x":"name",
"ymin":"score_min",
"ymax":"score_max"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"errorbar",
"data":{
}
}],
"theme":{
"flavor":"darcula"
},
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"name"
},{
"type":"float",
"column":"score"
},{
"type":"float",
"column":"score_min"
},{
"type":"float",
"column":"score_max"
}]
},
"spec_id":"183"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 DeflaterDefaultLevelBenchmark 
 
 
 
 
 
 
 
 
 DeflaterMaxLevelBenchmark 
 
 
 
 
 
 
 
 
 DeflaterMinLevelBenchmark 
 
 
 
 
 
 
 
 
 
 
 0.00 
 
 
 
 
 
 
 5.00 
 
 
 
 
 
 
 10.00 
 
 
 
 
 
 
 15.00 
 
 
 
 
 
 
 20.00 
 
 
 
 
 
 
 25.00 
 
 
 
 
 
 
 30.00 
 
 
 
 
 
 
 35.00 
 
 
 
 
 
 
 40.00 
 
 
 
 
 
 
 45.00 
 
 
 
 
 
 
 50.00 
 
 
 
 
 
 
 55.00 
 
 
 
 
 
 
 60.00 
 
 
 
 
 
 
 65.00 
 
 
 
 
 
 
 70.00 
 
 
 
 
 
 
 75.00 
 
 
 
 
 
 
 80.00 
 
 
 
 
 
 
 85.00 
 
 
 
 
 
 
 
 
 deflate (wasmJs) 
 
 
 
 
 MiB/s 
 
 
 
 
 Implementation 
 
 
 
 
 
 
 
 
 name 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 DeflaterDefaultLevelBenchmark 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 DeflaterMaxLevelBenchmark 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 DeflaterMin

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="SdKob8" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1200.0, 
 height: 800.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("SdKob8");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"deflate (jvm)"
},
"mapping":{
},
"guides":{
"x":{
"title":"Implementation"
},
"y":{
"title":"MiB/s"
}
},
"data":{
"score":[332.67932734122826,150.6438179030474,354.4681702624746,150.61033188439455,376.86779559282786,603.3493649839543],
"name":["DeflaterDefaultLevelBenchmark","JvmDeflaterDefaultLevelBenchmark","DeflaterMaxLevelBenchmark","JvmDeflaterMaxLevelBenchmark","DeflaterMinLevelBenchmark","JvmDeflaterMinLevelBenchmark"],
"score_max":[350.51096541473385,151.15475147714648,360.12947358358076,151.73672634666667,398.2762423849246,610.916065429316],
"score_min":[314.84768926772267,150.1328843289483,348.8068669413684,149.48393742212244,355.4593488007311,595.7826645385926]
},
"ggsize":{
"width":1200.0,
"height":800.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"format":".2f",
"limits":[null,null]
},{
"aesthetic":"fill",
"discrete":true
},{
"aesthetic":"x",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"name",
"y":"score",
"fill":"name"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data":{
}
},{
"mapping":{
"x":"name",
"ymin":"score_min",
"ymax":"score_max"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"errorbar",
"data":{
}
}],
"theme":{
"flavor":"darcula"
},
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"name"
},{
"type":"float",
"column":"score"
},{
"type":"float",
"column":"score_min"
},{
"type":"float",
"column":"score_max"
}]
},
"spec_id":"187"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 DeflaterDefaultLevelBenchmark 
 
 
 
 
 
 
 
 
 JvmDeflaterDefaultLevelBenchmark 
 
 
 
 
 
 
 
 
 DeflaterMaxLevelBenchmark 
 
 
 
 
 
 
 
 
 JvmDeflaterMaxLevelBenchmark 
 
 
 
 
 
 
 
 
 DeflaterMinLevelBenchmark 
 
 
 
 
 
 
 
 
 JvmDeflaterMinLevelBenchmark 
 
 
 
 
 
 
 
 
 
 
 0.00 
 
 
 
 
 
 
 50.00 
 
 
 
 
 
 
 100.00 
 
 
 
 
 
 
 150.00 
 
 
 
 
 
 
 200.00 
 
 
 
 
 
 
 250.00 
 
 
 
 
 
 
 300.00 
 
 
 

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="ftfhpS" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1200.0, 
 height: 800.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("ftfhpS");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"deflate (wasmWasi)"
},
"mapping":{
},
"guides":{
"x":{
"title":"Implementation"
},
"y":{
"title":"MiB/s"
}
},
"data":{
"score":[84.02058061251842,82.01220213015662,82.64382582407875],
"name":["DeflaterDefaultLevelBenchmark","DeflaterMaxLevelBenchmark","DeflaterMinLevelBenchmark"],
"score_max":[84.74749936991999,82.71685563508032,83.73117607732273],
"score_min":[83.29366185511684,81.30754862523293,81.55647557083476]
},
"ggsize":{
"width":1200.0,
"height":800.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"format":".2f",
"limits":[null,null]
},{
"aesthetic":"fill",
"discrete":true
},{
"aesthetic":"x",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"name",
"y":"score",
"fill":"name"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data":{
}
},{
"mapping":{
"x":"name",
"ymin":"score_min",
"ymax":"score_max"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"errorbar",
"data":{
}
}],
"theme":{
"flavor":"darcula"
},
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"name"
},{
"type":"float",
"column":"score"
},{
"type":"float",
"column":"score_min"
},{
"type":"float",
"column":"score_max"
}]
},
"spec_id":"191"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 DeflaterDefaultLevelBenchmark 
 
 
 
 
 
 
 
 
 DeflaterMaxLevelBenchmark 
 
 
 
 
 
 
 
 
 DeflaterMinLevelBenchmark 
 
 
 
 
 
 
 
 
 
 
 0.00 
 
 
 
 
 
 
 5.00 
 
 
 
 
 
 
 10.00 
 
 
 
 
 
 
 15.00 
 
 
 
 
 
 
 20.00 
 
 
 
 
 
 
 25.00 
 
 
 
 
 
 
 30.00 
 
 
 
 
 
 
 35.00 
 
 
 
 
 
 
 40.00 
 
 
 
 
 
 
 45.00 
 
 
 
 
 
 
 50.00 
 
 
 
 
 
 
 55.00 
 
 
 
 
 
 
 60.00 
 
 
 
 
 
 
 65.00 
 
 
 
 
 
 
 70.00 
 
 
 
 
 
 
 75.00 
 
 
 
 
 
 
 80.00 
 
 
 
 
 
 
 85.00 
 
 
 
 
 
 
 
 
 deflate (wasmWasi) 
 
 
 
 
 MiB/s 
 
 
 
 
 Implementation 
 
 
 
 
 
 
 
 
 name 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 DeflaterDefaultLevelBenchmark 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 DeflaterMaxLevelBenchmark 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 Defl

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="JCWtPH" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1200.0, 
 height: 800.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("JCWtPH");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"inflate (linuxX64)"
},
"mapping":{
},
"guides":{
"x":{
"title":"Implementation"
},
"y":{
"title":"MiB/s"
}
},
"data":{
"score":[596.567939492461,346.9746771333732,596.6733972848454,347.2618462481631,538.5486367336782,335.64063354259133],
"name":["InflaterDefaultLevelBenchmark","NativeInflaterDefaultLevelBenchmark","InflaterMaxLevelBenchmark","NativeInflaterMaxLevelBenchmark","InflaterMinLevelBenchmark","NativeInflaterMinLevelBenchmark"],
"score_max":[600.6763848739536,350.4876301201097,613.504325202826,352.6462318996457,548.6325518252075,339.17707879322575],
"score_min":[592.4594941109685,343.46172414663664,579.8424693668647,341.8774605966805,528.4647216421489,332.1041882919569]
},
"ggsize":{
"width":1200.0,
"height":800.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"format":".2f",
"limits":[null,null]
},{
"aesthetic":"fill",
"discrete":true
},{
"aesthetic":"x",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"name",
"y":"score",
"fill":"name"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data":{
}
},{
"mapping":{
"x":"name",
"ymin":"score_min",
"ymax":"score_max"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"errorbar",
"data":{
}
}],
"theme":{
"flavor":"darcula"
},
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"name"
},{
"type":"float",
"column":"score"
},{
"type":"float",
"column":"score_min"
},{
"type":"float",
"column":"score_max"
}]
},
"spec_id":"195"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 InflaterDefaultLevelBenchmark 
 
 
 
 
 
 
 
 
 NativeInflaterDefaultLevelBenchmark 
 
 
 
 
 
 
 
 
 InflaterMaxLevelBenchmark 
 
 
 
 
 
 
 
 
 NativeInflaterMaxLevelBenchmark 
 
 
 
 
 
 
 
 
 InflaterMinLevelBenchmark 
 
 
 
 
 
 
 
 
 NativeInflaterMinLevelBenchmark 
 
 
 
 
 
 
 
 
 
 
 0.00 
 
 
 
 
 
 
 50.00 
 
 
 
 
 
 
 100.00 
 
 
 
 
 
 
 150.00 
 
 
 
 
 
 
 200.00 
 
 
 
 
 
 
 250.00 
 
 
 
 
 


<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="yEetxN" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1200.0, 
 height: 800.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("yEetxN");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"inflate (wasmJs)"
},
"mapping":{
},
"guides":{
"x":{
"title":"Implementation"
},
"y":{
"title":"MiB/s"
}
},
"data":{
"score":[532.5625934143898,528.3379856986719,488.24197753263917],
"name":["InflaterDefaultLevelBenchmark","InflaterMaxLevelBenchmark","InflaterMinLevelBenchmark"],
"score_max":[540.8374720661509,536.408202429151,497.13997565413484],
"score_min":[524.2877147626286,520.2677689681929,479.3439794111435]
},
"ggsize":{
"width":1200.0,
"height":800.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"format":".2f",
"limits":[null,null]
},{
"aesthetic":"fill",
"discrete":true
},{
"aesthetic":"x",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"name",
"y":"score",
"fill":"name"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data":{
}
},{
"mapping":{
"x":"name",
"ymin":"score_min",
"ymax":"score_max"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"errorbar",
"data":{
}
}],
"theme":{
"flavor":"darcula"
},
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"name"
},{
"type":"float",
"column":"score"
},{
"type":"float",
"column":"score_min"
},{
"type":"float",
"column":"score_max"
}]
},
"spec_id":"199"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 InflaterDefaultLevelBenchmark 
 
 
 
 
 
 
 
 
 InflaterMaxLevelBenchmark 
 
 
 
 
 
 
 
 
 InflaterMinLevelBenchmark 
 
 
 
 
 
 
 
 
 
 
 0.00 
 
 
 
 
 
 
 50.00 
 
 
 
 
 
 
 100.00 
 
 
 
 
 
 
 150.00 
 
 
 
 
 
 
 200.00 
 
 
 
 
 
 
 250.00 
 
 
 
 
 
 
 300.00 
 
 
 
 
 
 
 350.00 
 
 
 
 
 
 
 400.00 
 
 
 
 
 
 
 450.00 
 
 
 
 
 
 
 500.00 
 
 
 
 
 
 
 550.00 
 
 
 
 
 
 
 
 
 inflate (wasmJs) 
 
 
 
 
 MiB/s 
 
 
 
 
 Implementation 
 
 
 
 
 
 
 
 
 name 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 InflaterDefaultLevelBenchmark 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 InflaterMaxLevelBenchmark 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 InflaterMinLevelBenchmark

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="sPV01G" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1200.0, 
 height: 800.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("sPV01G");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"inflate (js)"
},
"mapping":{
},
"guides":{
"x":{
"title":"Implementation"
},
"y":{
"title":"MiB/s"
}
},
"data":{
"score":[218.2390524517836,215.25037685194062,109.81342729537496],
"name":["InflaterDefaultLevelBenchmark","InflaterMaxLevelBenchmark","InflaterMinLevelBenchmark"],
"score_max":[220.0376412091125,220.48325330498272,111.04293546307554],
"score_min":[216.4404636944547,210.0175003988985,108.58391912767439]
},
"ggsize":{
"width":1200.0,
"height":800.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"format":".2f",
"limits":[null,null]
},{
"aesthetic":"fill",
"discrete":true
},{
"aesthetic":"x",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"name",
"y":"score",
"fill":"name"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data":{
}
},{
"mapping":{
"x":"name",
"ymin":"score_min",
"ymax":"score_max"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"errorbar",
"data":{
}
}],
"theme":{
"flavor":"darcula"
},
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"name"
},{
"type":"float",
"column":"score"
},{
"type":"float",
"column":"score_min"
},{
"type":"float",
"column":"score_max"
}]
},
"spec_id":"203"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 InflaterDefaultLevelBenchmark 
 
 
 
 
 
 
 
 
 InflaterMaxLevelBenchmark 
 
 
 
 
 
 
 
 
 InflaterMinLevelBenchmark 
 
 
 
 
 
 
 
 
 
 
 0.00 
 
 
 
 
 
 
 20.00 
 
 
 
 
 
 
 40.00 
 
 
 
 
 
 
 60.00 
 
 
 
 
 
 
 80.00 
 
 
 
 
 
 
 100.00 
 
 
 
 
 
 
 120.00 
 
 
 
 
 
 
 140.00 
 
 
 
 
 
 
 160.00 
 
 
 
 
 
 
 180.00 
 
 
 
 
 
 
 200.00 
 
 
 
 
 
 
 220.00 
 
 
 
 
 
 
 
 
 inflate (js) 
 
 
 
 
 MiB/s 
 
 
 
 
 Implementation 
 
 
 
 
 
 
 
 
 name 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 InflaterDefaultLevelBenchmark 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 InflaterMaxLevelBenchmark 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 InflaterMinLevelBenchmark

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="l3NMWO" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1200.0, 
 height: 800.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("l3NMWO");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"inflate (jvm)"
},
"mapping":{
},
"guides":{
"x":{
"title":"Implementation"
},
"y":{
"title":"MiB/s"
}
},
"data":{
"score":[2033.561190555539,446.6982148468869,1997.391652909881,444.88462284608966,1837.660608867027,422.19484433968466],
"name":["InflaterDefaultLevelBenchmark","JvmInflaterDefaultLevelBenchmark","InflaterMaxLevelBenchmark","JvmInflaterMaxLevelBenchmark","InflaterMinLevelBenchmark","JvmInflaterMinLevelBenchmark"],
"score_max":[2074.4349045519916,448.35188992572904,2020.1520508823457,447.7119156002536,1850.3507551196624,483.8402722169333],
"score_min":[1992.6874765590862,445.0445397680448,1974.6312549374163,442.05733009192573,1824.9704626143914,360.54941646243606]
},
"ggsize":{
"width":1200.0,
"height":800.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"format":".2f",
"limits":[null,null]
},{
"aesthetic":"fill",
"discrete":true
},{
"aesthetic":"x",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"name",
"y":"score",
"fill":"name"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data":{
}
},{
"mapping":{
"x":"name",
"ymin":"score_min",
"ymax":"score_max"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"errorbar",
"data":{
}
}],
"theme":{
"flavor":"darcula"
},
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"name"
},{
"type":"float",
"column":"score"
},{
"type":"float",
"column":"score_min"
},{
"type":"float",
"column":"score_max"
}]
},
"spec_id":"207"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 InflaterDefaultLevelBenchmark 
 
 
 
 
 
 
 
 
 JvmInflaterDefaultLevelBenchmark 
 
 
 
 
 
 
 
 
 InflaterMaxLevelBenchmark 
 
 
 
 
 
 
 
 
 JvmInflaterMaxLevelBenchmark 
 
 
 
 
 
 
 
 
 InflaterMinLevelBenchmark 
 
 
 
 
 
 
 
 
 JvmInflaterMinLevelBenchmark 
 
 
 
 
 
 
 
 
 
 
 0.00 
 
 
 
 
 
 
 200.00 
 
 
 
 
 
 
 400.00 
 
 
 
 
 
 
 600.00 
 
 
 
 
 
 
 800.00 
 
 
 
 
 
 
 1000.00 
 
 
 
 
 
 
 1200.00 
 
 
 
 

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="khW5vV" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1200.0, 
 height: 800.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("khW5vV");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"inflate (wasmWasi)"
},
"mapping":{
},
"guides":{
"x":{
"title":"Implementation"
},
"y":{
"title":"MiB/s"
}
},
"data":{
"score":[524.4873784735669,527.1240912450271,488.53577974392886],
"name":["InflaterDefaultLevelBenchmark","InflaterMaxLevelBenchmark","InflaterMinLevelBenchmark"],
"score_max":[531.9723149625156,531.8322932685431,494.61396582273187],
"score_min":[517.0024419846181,522.4158892215111,482.45759366512584]
},
"ggsize":{
"width":1200.0,
"height":800.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"format":".2f",
"limits":[null,null]
},{
"aesthetic":"fill",
"discrete":true
},{
"aesthetic":"x",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"name",
"y":"score",
"fill":"name"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data":{
}
},{
"mapping":{
"x":"name",
"ymin":"score_min",
"ymax":"score_max"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"errorbar",
"data":{
}
}],
"theme":{
"flavor":"darcula"
},
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"name"
},{
"type":"float",
"column":"score"
},{
"type":"float",
"column":"score_min"
},{
"type":"float",
"column":"score_max"
}]
},
"spec_id":"211"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 InflaterDefaultLevelBenchmark 
 
 
 
 
 
 
 
 
 InflaterMaxLevelBenchmark 
 
 
 
 
 
 
 
 
 InflaterMinLevelBenchmark 
 
 
 
 
 
 
 
 
 
 
 0.00 
 
 
 
 
 
 
 50.00 
 
 
 
 
 
 
 100.00 
 
 
 
 
 
 
 150.00 
 
 
 
 
 
 
 200.00 
 
 
 
 
 
 
 250.00 
 
 
 
 
 
 
 300.00 
 
 
 
 
 
 
 350.00 
 
 
 
 
 
 
 400.00 
 
 
 
 
 
 
 450.00 
 
 
 
 
 
 
 500.00 
 
 
 
 
 
 
 550.00 
 
 
 
 
 
 
 
 
 inflate (wasmWasi) 
 
 
 
 
 MiB/s 
 
 
 
 
 Implementation 
 
 
 
 
 
 
 
 
 name 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 InflaterDefaultLevelBenchmark 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 InflaterMaxLevelBenchmark 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 InflaterMinLevelBenchmark